# Calibrated Data Check Observer's Notebook

This notebook performs various checks on the UHF data that has been flagged and calibrated by the museek pipeline.

# The link for GoogleForms:
https://docs.google.com/forms/d/e/1FAIpQLSc3rmH_jAhCU2TEk9_eYmmC44LiMA27DcG_X7NtrDcf0CB7EA/viewform

In [ ]:
# -- Import libraries --
# Standard libraries
import logging
import random
from functools import reduce
from pathlib import Path

# Astro libraries
import healpy as hp
import matplotlib as mpl
import numpy as np
import numpy.typing as npt
import pysm3
import xarray as xr
from astropy import units as u
from astropy.coordinates import SkyCoord
from matplotlib import pyplot as plt
from matplotlib import ticker
from matplotlib.offsetbox import AnchoredText
from scipy.stats import spearmanr
from sklearn.linear_model import HuberRegressor

# Museek libraries
from museek.util import notebook_helper

In [ ]:
# -- Plot, logging, notebook display configuration --
# Update matplotlib default parameters
mpl_params = {
    # Set font family to Sans-serif for better readability on computer
    "font.family": "sans-serif",
    # Optional: Specify preferred sans-serif fonts (fallback order)
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 11,
    # Make sure the figure background is white
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    # Always turn on minor ticks
    "ytick.minor.visible": True,
    "xtick.minor.visible": True,
}
mpl.rcParams.update(mpl_params)

# Configure xarray to expand data variables by default when displaying datasets to
# make sure all variables are displayed when converting the notebook to HTML
xr.set_options(display_expand_data_vars=True, display_max_rows=200)

# Setup logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    logger.addHandler(handler)

# Suppress healpy and pysm3 verbosity to WARNING only
logging.getLogger("pysm3").setLevel(logging.WARNING)
logging.getLogger("healpy").setLevel(logging.WARNING)

In [ ]:
# Define some global variables and functiosn to help with plotting
BBOX = dict(boxstyle="round,pad=0.8", facecolor="white", alpha=0.5)


def add_anchored_text(ax, text, loc="upper left", alpha=0.5):
    """Add anchored text to a given axis."""
    anchored_text = AnchoredText(
        text,
        loc=loc,  # Exact same location strings as legend
        borderpad=0.5,  # Fixed physical gap from axis edge
        frameon=True,  # Set False if you want a invisible background box
    )
    anchored_text.patch.set_alpha(alpha)
    anchored_text.patch.set_facecolor("white")
    anchored_text.patch.set_edgecolor("lightgrey")

    ax.add_artist(anchored_text)

## 1. Data Preparation

Read scan and calibrated visibility and flags from `aoflagger_plugin_postcalibration.pickle` and build [xarray](https://xarray.dev/) datasets to standardise data format while minimising memory footprint.

In [ ]:
# Cell tags: parameters
# -- Default parameters --
# This cell has "parameters" tag with default data and path values neccesary to run
# the notebook. Papermill will use these values when executing the notebook if no
# parameter overridden is passed through the papermill command. If parameter overridden
# is provided, a new cell will be injected below this cell with the desired parameters,
# overwritting values in this cell. These default parameters can also be inspected by
# calling `papermill --help-notebook <path/to/this/notebook>`
block_name: str = "1779036710"  # Block name (CBID)
patch: str = "box14"  # Patch name, e.g. "box14"
base_context_folder: str = "/idia/projects/meerklass/MEERKLASS-1/museek"  # Base context folder. Notebook will look for the data in base_context_folder/patch/block_name/context

In [ ]:
# Build context file path.
context_dir = Path(base_context_folder) / patch / block_name / "context"
context_file = context_dir / "aoflagger_plugin_postcalibration.pickle"

# Load context data into an xarray Dataset, trimming frequency dimension.
# This should only use as much memory as the file size + notebook overhead.
# Apart from minimising memory footprint, putting the context data into xarray
# standardises the data structure as the pickle file contains a mixture of several
# data formats and some variables are sometimes not saved in the most logical places.
#
# If manually executing this notebook for testing, it is recommded to define and pass
# a `cache_file` parameter to `load_context_to_ds`, which will write the constructed
# datasets to `.nc` file on disk for faster subsequent loading. xarray dataset is
# lazy-loaded -- only metadata will be read into memory at the load time, and data
# variables will only be loaded into memory when accessed. This can help reduce memory
# usage greatly when experimenting with calculations. `skip_cache=True` can be used to
# force reloading the context data from the pickle file and overwriting the cache file.
#
# cache_file = Path(os.getenv("XDG_CACHE_HOME")) / f"ds_cache_{block_name}.nc"
# ds = notebook_helper.load_context_to_ds(
#     context_file, frequency_range="auto", cache_file=cache_file, skip_cache=False
# )
#
ds = notebook_helper.load_context_to_ds(context_file, frequency_range="auto")

In [ ]:
# Print the dataset summary
print(ds)

## 4. Antenna Usability, Flag Fraction and Correlation with Synchrotron Model

Calculate antenna usability after each plugin, and the final usable antennas after
the last calibrated step. This is done by accumulating the flags from each plugin and
checking which antennas are fully flagged

In [ ]:
def compute_usability(ds) -> list[str]:
    # Memory strategy:
    #   - Keep a single (timestamps, frequencies, antennas) bool array in memory.
    #   - Feeds-dim flags are immediately collapsed to antennas (OR over H/V)
    #     before accumulation, halving the array size vs. keeping the feeds dim.
    #   - Accumulate with numpy in-place |= to avoid any allocation per step.
    #   - Only one temporary array (the per-step antenna-collapsed flag) exists
    #     alongside the cumulative; it is freed at the start of the next iteration.
    #

    meerkat_all_ants = ds.attrs["all_meerkat_antennas"]
    # Empty boolean array for storing cumulative flags. This is the same shape as
    # cal_flags_combined (excluding its size-1 polarisations dim), which is
    # (timestamps, frequencies, antennas)
    cumulative = np.zeros(
        ds["cal_flags_combined"].isel(polarisations=0).shape, dtype=bool
    )

    # List for storing result, (plugin_name, flag_name, usable_count)
    flag_summary = []

    # Iterate over each raw flag in the raw_flag_name_list, collapsing H/V to antennas
    for flag_name in ds.attrs["raw_flag_name_list"]:
        flag_da = ds[f"raw_flags_{flag_name}"]

        # If a feeds dim is present, OR-collapse H/V to antennas
        if "feeds" in flag_da.dims:
            flag_vals = flag_da.any(dim="feeds").values
        else:
            flag_vals = flag_da.values

        # Cumulate flags in-place with OR
        cumulative = cumulative | flag_vals

        # Collasing (timestamps, frequencies,) to yield (antennas,) flags
        per_ant_flags = cumulative.all(axis=(0, 1))

        # Sum over (antennas,) of the reversed flags to get the number of usable antennas
        num_usable_ants = int(np.sum(np.logical_not(per_ant_flags)))

        # Storing result
        plugin = notebook_helper.FLAG_PLUGIN_MAP.get(flag_name, flag_name)
        flag_summary.append((plugin, flag_name, num_usable_ants))
    del cumulative

    # For cal flags, we only need the number after the final calibration
    per_ant_flags_cal = notebook_helper.reduce_flags(
        flag_da=ds["cal_flags_combined"], output_dims=("antennas",), operator="and"
    ).values
    num_usable_ants = int(np.sum(np.logical_not(per_ant_flags_cal)))
    flag_summary.append(
        ("aoflagger_postcalibration_plugin", "cal_flags_combined", num_usable_ants)
    )

    # Print the names of antennas excluded in this observation at scheduled time
    # (not in the antennas list)
    ant_not_in_observation = [a for a in meerkat_all_ants if a not in ds.antennas]
    print("Total numbers of MeerKAT antennas:", len(meerkat_all_ants))
    print(
        "Antennas excluded in this observation at scheduled time:",
        len(ant_not_in_observation),
        ant_not_in_observation,
    )

    # Print result. Some plugins add two flags (e.g. antenna_flagger_plugin).
    # Print one row per plugin (antenna_flagger_plugin adds two flags — suppress the first).
    print("Numbers of usable antennas after each pipeline step:")
    for i, (plugin, flag_name, count) in enumerate(flag_summary):
        is_last_for_plugin = (
            i == len(flag_summary) - 1 or flag_summary[i + 1][0] != plugin
        )
        if is_last_for_plugin:
            print(f"  {plugin}: {count}")

    # Print the names of antennas fully masked in the final calibrated_vis
    # (after last calibration step)
    ants_flagged_final = ds.antennas[per_ant_flags_cal].values.tolist()
    print(
        "Antennas flagged after the last calibration step:",
        len(ants_flagged_final),
        ants_flagged_final,
    )
    good_ants = [a for a in ds.antennas.values.tolist() if a not in ants_flagged_final]
    return good_ants

In [ ]:
def plot_flag_fraction_and_r_vis(ds) -> None:
    """
    Plot the per-antenna flag fraction and correlation coefficient with Synchrotron
    model after the final calibration step.
    """
    # Calculate / extract flag fraction and correlation coefficient with Synchrotron model
    # This has dims=("antennas",)
    flag_fraction = ds["cal_flags_combined"].sum(dim=["timestamps", "frequencies"]).sel(
        polarisations="I", drop=True
    ) / (ds.sizes["timestamps"] * ds.sizes["frequencies"])
    r_vis = ds["r_vis_synch_ant"]

    # We want to make plots with all antennas on the x-axis. Use xarray `reindex` to expand
    # "antennas" dimension to include all antennas, filling missing values with NaN.
    all_ants = ds.attrs["all_meerkat_antennas"]
    flag_fraction = flag_fraction.reindex(antennas=all_ants, fill_value=np.nan)
    r_vis = r_vis.reindex(antennas=all_ants, fill_value=np.nan)

    # Plot the data. Use rasterized=True to reduce file size when saving.
    # Also use a custom numeric antenna numbers as x to help with plotting the STD
    # shade region beyond the range of antennas.
    x = np.arange(0, 66)
    fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex="all", layout="constrained")
    for ax, y in zip(axes, [flag_fraction, r_vis]):
        # First plot mean +/- std
        # The std spance 0 to
        ax.fill_between(
            x=x,
            y1=y.mean(skipna=True) - y.std(skipna=True),
            y2=y.mean(skipna=True) + y.std(skipna=True),
            color="C0",
            alpha=0.15,
            label="Mean ± σ",
        )
        ax.axhline(y=y.mean(skipna=True), color="C0", linestyle="--", label="Mean")
        ax.scatter(x=x[1:-1], y=y.values, rasterized=True)

        # Deal with grid and ticks. We want no minor ticks on the x-axis.
        ax.xaxis.set_minor_locator(ticker.NullLocator())
        ax.grid(visible=True, which="major", axis="x", linestyle=":")
    axes[-1].set_xlim(0, 64)
    # Relabel the x-axis with antenna names
    axes[-1].set_xticks(
        x,
        [
            "",
        ]
        + all_ants
        + [""],
        rotation=90,
        fontsize="small",
    )
    # Recolor antennas not in the observation in grey
    ant_not_in_observation = [a for a in all_ants if a not in ds.antennas]
    for label in axes[-1].get_xticklabels():
        if label.get_text() in ant_not_in_observation:
            label.set_color("grey")
    fig.text(
        0.01,
        0.01,
        "Antennas excluded from this observation at scheduled time",
        fontsize="x-small",
        color="grey",
        ha="left",
    )
    axes[0].set_ylabel("Flag Fraction")
    axes[1].set_ylabel("Correlation Coefficient")
    axes[1].set_xlabel("Antenna Names")
    fig.suptitle(f"{ds.attrs['block_name']}")
    fig.align_ylabels(axes)
    axes[0].legend()

In [ ]:
plot_flag_fraction_and_r_vis(ds)

In [ ]:
good_ants = compute_usability(ds)

## 6. Non-linearity contamination

Evaluate correlation between stripy pattern and the GSM RFI to quantify non-linearty contamination.
Caveat: works better if the sky model used match the sych emission in the patch

In [ ]:
def ch_finder(frequency_mhz: float, freq_arr: np.ndarray) -> int:
    ch1 = np.where(freq_arr < frequency_mhz)[0][-1]
    ch2 = np.where(freq_arr > frequency_mhz)[0][0]
    return (
        ch1
        if abs(freq_arr[ch1] - frequency_mhz) <= abs(freq_arr[ch2] - frequency_mhz)
        else ch2
    )

In [ ]:
def zebra_correlation_test(
    ds: xr.Dataset, freq_plot: float, sky_model: np.ndarray, ch_gsm1: int, ch_gsm2: int
) -> tuple[npt.NDArray[float]]:
    """Return Spearman correlation array of shape (n_antennas, n_feeds, 2)."""
    freq_arr = ds.coords["frequencies"].values
    antennas = ds.coords["antennas"].values
    ra_arr = ds["ra"].values
    dec_arr = ds["dec"].values
    raw = ds["raw_vis"].values
    nd_flag = ds["raw_flags_noise_diode_on"].values
    ch_plot = ch_finder(freq_plot, freq_arr)
    nd_times = nd_flag[:, ch_plot, 0, 0]
    gsm = np.nansum(raw[~nd_times, ch_gsm1:ch_gsm2, :, :], axis=1)
    spearman = np.zeros((len(antennas), 2, 2), dtype=float)
    for i_ant in range(len(antennas)):
        c = SkyCoord(
            ra=ra_arr[:, i_ant] * u.degree,
            dec=dec_arr[:, i_ant] * u.degree,
            frame="icrs",
        )
        theta = (np.pi / 2) - c.galactic.b.rad
        phi = c.galactic.l.rad
        synch_I = hp.pixelfunc.get_interp_val(sky_model, theta, phi).value / 1e6
        for i_feed in range(2):
            y = raw[:, ch_plot, i_ant, i_feed]
            valid = ~nd_times & np.isfinite(
                y
            )  # (timestamps, frequencies, antennas, feeds)
            huber = HuberRegressor(
                epsilon=1.35,
                max_iter=100,
                alpha=0.0001,
                warm_start=False,
                fit_intercept=True,
                tol=1.0,
            ).fit(synch_I[valid, None], y[valid])
            ysub = y[valid] - synch_I[valid] * huber.coef_[0]
            spearman[i_ant, i_feed, :] = spearmanr(
                gsm[~nd_times[valid], i_ant, i_feed], ysub
            )  # noise-diode state at the plot frequency

    # Split into [H, V, mean] and return as a tuple
    spearman_h = spearman[:, 0, 0]
    spearman_v = spearman[:, 1, 0]
    spearman_mean = spearman[:, :, 0].mean(axis=1)
    return spearman_h, spearman_v, spearman_mean

In [ ]:
nside = 128
beamsize = 57.5
beam_ref_freq = 1500.0
freq_plot = 730.0
sky = pysm3.Sky(nside=nside, preset_strings=["s1"])
map = sky.get_emission(freq_plot * u.MHz).value
map_ref_smooth = pysm3.apply_smoothing_and_coord_transform(
    map,
    fwhm=beamsize
    * u.arcmin
    * (beam_ref_freq * u.MHz / (freq_plot * u.MHz)).decompose().value,
)
# --- Synchrotron model setup ---
sky_model = map_ref_smooth[0]
# GSM 900 downlink band (925–960 MHz) used as the RFI proxy for the zebra test.
GSM_DOWNLINK_MHZ = (925, 960)
ch_gsm1 = ch_finder(GSM_DOWNLINK_MHZ[0], ds.frequencies.values)
ch_gsm2 = ch_finder(GSM_DOWNLINK_MHZ[1], ds.frequencies.values)

spearman_skysub = zebra_correlation_test(ds, freq_plot, sky_model, ch_gsm1, ch_gsm2)

In [ ]:
def plot_zebra_test(ds: xr.Dataset, spearman: tuple[npt.NDArray]) -> None:
    x = ds.antennas

    panel_labels = ["H", "V", "H & V Mean"]
    gridspec_kw = {"width_ratios": [10, 1], "hspace": 0.0, "wspace": 0.0}
    fig, axes = plt.subplots(
        3,
        2,
        figsize=(10, 6),
        sharex="col",
        sharey="row",
        gridspec_kw=gridspec_kw,
        layout="constrained",
    )

    for i, (ax, y, label) in enumerate(zip(axes, spearman, panel_labels)):
        # Left column - main scatter plot
        ax[0].scatter(x, y, c=f"C{i}", marker="x")
        ax[0].axhline(np.mean(y), color="grey", linestyle="--", label="Mean")
        # Turn off minor ticks on the x-axis
        ax[0].xaxis.set_minor_locator(ticker.NullLocator())
        ax[0].grid(visible=True, which="major", axis="x", linestyle=":")
        ax[0].set_ylabel(f"Values\n({label})")
        # Print mean and STD for each panel
        mean_val = np.nanmean(y)
        std_val = np.nanstd(y)
        text = f"Mean: {mean_val:.3f}\nSTD: {std_val:.3f}"
        add_anchored_text(ax[0], text, loc="lower right")
        # ax[0].text(1.0, 0.0, f"Mean: {mean_val:.3f}\nSTD: {std_val:.3f}", transform=ax[0].transAxes, ha="right", va="bottom", bbox=BBOX)

        # Plot half-violin (KDE) on the right
        ax[1].violinplot(
            y,
            positions=[0],
            side="high",
            showmeans=False,
            showextrema=False,
            facecolor=f"C{i}",
        )
        ax[1].axhline(np.mean(y), color="grey", linestyle="--", label="Mean")
        # Hide the right, top and bottom spines (borders)
        ax[1].get_xaxis().set_visible(False)
        for spine in ["top", "bottom", "right"]:
            ax[1].spines[spine].set_visible(False)
    # Set x-axis ticks and labels for the bottom row of scatter plots
    axes[-1, 0].tick_params(axis="x", which="major", rotation=90, labelsize="x-small")
    axes[-1, 0].set_xlabel("Antennas")
    fig.align_ylabels(axes[:, 0])
    fig.suptitle(f"Raw Map and GSM Correlation - {block_name}")
    # print(
    #     f"H    : mean = {np.nanmean(spearman_h):.3f}, std = {np.nanstd(spearman_h):.3f}"
    # )
    # print(
    #     f"V    : mean = {np.nanmean(spearman_v):.3f}, std = {np.nanstd(spearman_v):.3f}"
    # )
    # print(
    #     f"Mean : mean = {np.nanmean(spearman_mean):.3f}, std = {np.nanstd(spearman_mean):.3f}"
    # )


plot_zebra_test(
    ds, spearman_skysub
)  # Top: per-antenna line plot  # Bottom: boxplot with overlaid scatter  # arcmin FWHM at beam_frequency  # MHz

## 7. Raw vs calibrated: frequency spectra

Raw vs. Calibrated Data

Examine the frequency median of the raw and calibrated data for all antennas to assess data quality and evaluate the effectiveness of RFI flagging after calibration and following post-calibration aoflagger processing.

In [ ]:
def calculate_median(
    ds: xr.Dataset, vis_key: str, dim: str | tuple[str], flag_key: str | None = None
) -> xr.DataArray:
    """Calculate the median of a visibility data variable along specified dimensions.

    Parameters
    ----------
    ds : xr.Dataset
        The input xarray Dataset containing visibility data.
    vis_key : str
        The key of the visibility data variable in the Dataset.
    dim : str or tuple of str
        The dimension(s) along which to calculate the median.
    flag_key : str, optional
        The key of the flag data variable in the Dataset to apply to the visibility
        data before calculating the median. If None, no flagging is applied.

    Returns
    -------
    xr.DataArray
        A new xarray DataArray containing the median values of the specified visibility
        data variable.
    """
    if flag_key is not None:
        flags = ds[flag_key]
        valid_data = ds[vis_key].where(~flags)
    else:
        valid_data = ds[vis_key]

    median_data = valid_data.median(dim=dim, skipna=True)
    return median_data

In [ ]:
def plot_raw_vs_calibrated_spectra(ds: xr.Dataset) -> None:
    """Plot time-averaged median spectra of raw and calibrated visibilities."""
    raw_timemedian = calculate_median(
        ds, vis_key="raw_vis", dim="timestamps", flag_key="raw_flags_combined"
    )  # (frequencies, antennas, feeds) — computed on the fly, freed when this returns.
    cal_timemedian = calculate_median(
        ds, vis_key="cal_vis", dim="timestamps", flag_key="cal_flags_combined"
    ).sel(polarisations="I", drop=True)  # (frequencies, antennas)
    freq_arr = raw_timemedian.coords["frequencies"]

    fig, (ax1, ax2, ax3) = plt.subplots(
        3, 1, figsize=(10, 8), sharex="all", layout="constrained"
    )
    ax1.plot(freq_arr, raw_timemedian.sel(feeds="h"), lw=0.7)
    ax2.plot(freq_arr, raw_timemedian.sel(feeds="v"), lw=0.7)
    ax3.plot(freq_arr, cal_timemedian, lw=0.7)
    ax3.xaxis.minorticks_on()
    for ax, label in [(ax1, "Raw HH"), (ax2, "Raw VV"), (ax3, "Calibrated")]:
        add_anchored_text(ax, label, loc="upper right")
    ax1.set_ylabel("Temperature\n[uncalibrated]")
    ax2.set_ylabel("Temperature\n[uncalibrated]")
    ax3.set_ylabel("Temperature\n[$K_{RJ}$]")
    ax3.set_xlabel("Frequency [MHz]")
    fig.suptitle(f"Raw and Calibrated Autos Time Medians - {block_name}")
    fig.align_ylabels()

In [ ]:
plot_raw_vs_calibrated_spectra(ds)

## 8. Before AOFlagger: spectra and time series

Examine the spectra of the raw visibilities before applying AOFlagger, and compare them with the data after flagging.
This allows us to assess the data quality and determine how much data has been flagged by SARAO or AOFlagger.

In [ ]:
# Build a mask for flags that existed before AOFlagger was applied.
# These four flag layers correspond to FLAG_NAME_LIST indices 0-3.
PRE_AOFLAGGER_FLAGS = [
    "raw_flags_SARAO",
    "raw_flags_noise_diode_on",
    "raw_flags_known_rfi",
    "raw_flags_rawdata_low_value",
]

# Use reduce and xr.logical_or to combine all pre-AOFlagger flags into a single
# boolean array. This has advatage over looping over each variable and applying |
# operator in that it can handle NaNs via options in xr.ufuncs.logical_or although
# we are not using this option here since the flags are all boolean.
flags_before_aoflagger = reduce(
    xr.ufuncs.logical_or, [ds[_name] for _name in PRE_AOFLAGGER_FLAGS]
)

In [ ]:
def plot_before_aoflagger(ds: xr.Dataset) -> None:
    raw_vis_before_ao = ds["raw_vis"].where(~flags_before_aoflagger)
    vis_timemedian = raw_vis_before_ao.median(dim="timestamps", skipna=True)
    vis_freqmedian = raw_vis_before_ao.median(dim="frequencies", skipna=True)
    del (
        raw_vis_before_ao
    )  # Compute medians on the fly; freed when this function returns.

    freq_arr = vis_timemedian.coords["frequencies"]
    ts = vis_freqmedian.coords["timestamps"]
    # Unix time is in running seconds; convert to minutes for plotting.
    t_min = (ts - ts.min()) / 60.0

    fig, axes = plt.subplots(2, 2, figsize=(15, 6), sharex="col", layout="constrained")
    for ax, feed in zip(axes, ["h", "v"]):
        ax[0].plot(freq_arr, vis_timemedian.sel(feeds=feed), lw=0.7)
        ax[1].plot(t_min, vis_freqmedian.sel(feeds=feed), lw=0.7)
        add_anchored_text(ax[0], f"{(feed * 2).upper()}", loc="upper right")
        add_anchored_text(ax[1], f"{(feed * 2).upper()}", loc="upper right")
    axes[1, 0].set_xlabel("Frequency [MHz]")
    axes[1, 1].set_xlabel("Time [min]")
    axes[0, 0].set_ylabel("Temperature [uncalibrated]", y=0)
    axes[0, 1].set_ylabel("Temperatyre [uncalibrated]", y=0)
    fig.suptitle(
        f"Raw Autos Time and Frequency Medians Before AOFlagger - {ds.attrs['block_name']}"
    )
    fig.align_ylabels()


plot_before_aoflagger(ds)

## 9. Raw Visibiliy -- Mean Subtracted Frequency Spectra

Examine the frequency median of raw data (with frequency average removed)
from several randomly selected antennas to assess consistency across antennas.

In [ ]:
def plot_raw_time_series(ds: xr.Dataset, n_freqs: int = 5, n_ants: int = 15) -> None:
    freq_da = ds.coords["frequencies"]
    selected_freqs = np.linspace(
        freq_da.min().values, freq_da.max().values, n_freqs
    )  # Evenly spaced frequencies to plot
    selected_ants = random.sample(
        ds.coords["antennas"].values.tolist(), n_ants
    )  # Randomly select antennas to plot
    ts = ds.coords["timestamps"]
    t_min = (ts - ts.min()) / 60.0
    # Apply the combined flags to the raw visibilities, selecting only the chosen
    # antennas and frequencies to keep memory usage minimal.
    vis_flagged = (
        ds["raw_vis"]
        .sel(antennas=selected_ants)
        .sel(frequencies=selected_freqs, method="nearest")
        .where(
            ~ds["raw_flags_combined"]
            .sel(antennas=selected_ants)
            .sel(frequencies=selected_freqs, method="nearest")
        )
    )
    # Subtract vis_flagged by its mean
    vis_flagged = vis_flagged - vis_flagged.mean(dim="timestamps", skipna=True)

    # Prepare to plot
    fig, axes = plt.subplots(
        len(selected_freqs),
        2,
        figsize=(15, 2 * len(selected_freqs)),
        sharex="all",
        sharey="row",
        layout="constrained",
    )
    # Loop over row (frequency) and plot
    for row, freq in enumerate(selected_freqs):
        for col, feed in enumerate(["h", "v"]):
            axes[row, col].plot(
                t_min, vis_flagged.sel(frequencies=freq, feeds=feed), lw=0.5
            )
            add_anchored_text(
                axes[row, col],
                f"{freq:.1f} MHz - {(feed * 2).upper()}",
                loc="upper left",
            )
    # Label axes at figure level
    fig.supylabel("Temperature [uncalibrated]")
    fig.supxlabel("Time [min]")
    fig.suptitle(f"Mean-subtracted Raw Autos - {ds.attrs['block_name']}")
    # Add a single legend for all antennas at the top of the figure. Constrained layout
    # does not work with figure legend yet so we have to adjust the padding
    fig.legend(
        handles=axes[0, 0].lines,
        labels=selected_ants,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.98),
        ncol=8,
        frameon=False,
    )
    fig.get_layout_engine().set(rect=[0, 0, 1, 0.95])


plot_raw_time_series(ds)

## 10. Raw maps before AOFlagger (H and V, freq median)

For each antenna, scatter plots show the sky pointing positions in RA and Dec coloured by the
median visibility before the AOFlagger step. 1 Jy and 5 Jy catalogue sources are overplotted.

In [ ]:
# -- Common functions and variables for map ploting --
def scatter_sky(
    ax: mpl.axes.Axes,
    ra_arr: np.ndarray,
    dec_arr: np.ndarray,
    values: np.ndarray,
    vmin: float | None = None,
    vmax: float | None = None,
    cbar: bool = False,
    point_sources: dict | None = None,
):
    """Plot sky values at (RA, Dec) coordinates with point sources overlaid.

    Returns the `PathCollection` scatter artist so callers can build their own
    (e.g. shared, multi-axes) colorbar instead of using `cbar=True`.
    """
    sc = ax.scatter(
        ra_arr,
        dec_arr,
        c=values,
        edgecolor="none",
        cmap="jet",
        vmin=vmin,
        vmax=vmax,
        rasterized=True,
    )
    if cbar:
        ax.get_figure().colorbar(sc, ax=ax, label="Temperature [uncalibrated]")
    if point_sources is not None:
        ax.scatter(
            point_sources["ra_1jy"],
            point_sources["dec_1jy"],
            facecolors="none",
            edgecolors="black",
            marker="s",
            s=40,
            rasterized=True,
        )
        ax.scatter(
            point_sources["ra_5jy"],
            point_sources["dec_5jy"],
            color="black",
            marker="+",
            s=70,
            rasterized=True,
        )
    return sc


def get_synchrotron_map(
    freq_plot: float, beamsize: float, beam_ref_freq: float
) -> npt.NDArray:
    # -- Build synchrotron model --
    sky = pysm3.Sky(nside=128, preset_strings=["s1"])
    map = sky.get_emission(freq_plot * u.MHz).value
    # smooth with Gaussian beam scaled to freq_plot
    fwhm = beamsize * u.arcmin * (beam_ref_freq * u.MHz / (freq_plot * u.MHz))
    map_smooth = pysm3.apply_smoothing_and_coord_transform(map, fwhm=fwhm)[0]
    return map_smooth


def get_healpix_interp_vals(healpix_map, ra, dec):
    """Get interpolated values from a Healpix map at given RA and Dec coordinates."""
    c = SkyCoord(ra=ra * u.degree, dec=dec * u.degree, frame="icrs")
    theta = (np.pi / 2) - c.galactic.b.rad  # type: ignore[operator]
    phi = c.galactic.l.rad  # type: ignore[union-attr]
    vals = hp.pixelfunc.get_interp_val(healpix_map, theta, phi).value
    return vals


def load_point_sources_for_ds(ds: xr.Dataset, antenna: str = "m002") -> dict:
    """Load 1 Jy and 5 Jy point sources within the RA/Dec bounding box actually
    covered by the (RA, Dec) range in the dataset.

    A "good" antenna should be specified to avoid outliers in the RA/Dec range
    due to bad pointing.
    """
    ra_center = ds["ra"].sel(antennas=antenna).mean().values
    dec_center = ds["dec"].sel(antennas=antenna).mean().values
    ra_min = ds["ra"].sel(antennas=antenna).min().values
    ra_max = ds["ra"].sel(antennas=antenna).max().values
    dec_min = ds["dec"].sel(antennas=antenna).min().values
    dec_max = ds["dec"].sel(antennas=antenna).max().values
    ra_radius = max(abs(ra_center - ra_min), abs(ra_center - ra_max))
    dec_radius = max(abs(dec_center - dec_min), abs(dec_center - dec_max))
    return notebook_helper.load_point_sources(
        ra_center=ra_center,
        dec_center=dec_center,
        ra_radius=ra_radius,
        dec_radius=dec_radius,
    )


ANTENNAS_PER_FIG = 8

In [ ]:
point_sources = load_point_sources_for_ds(ds, antenna=good_ants[0])

In [ ]:
def plot_raw_maps(ds: xr.Dataset, flags: xr.DataArray, point_sources: dict) -> None:
    vis_freqmedian = ds["raw_vis"].where(~flags).median(dim="frequencies", skipna=True)
    ants = ds.coords["antennas"].values
    all_ants = ds.attrs["all_meerkat_antennas"]
    freq_mean = ds.coords["frequencies"].mean().values

    # Loop over group of antennas to plot in batches
    for fig_start in range(0, 64, ANTENNAS_PER_FIG):
        # Antennas for this batch
        batch = all_ants[fig_start : fig_start + ANTENNAS_PER_FIG]

        fig, axes = plt.subplots(
            ANTENNAS_PER_FIG,
            2,
            figsize=(10, 2 * ANTENNAS_PER_FIG),
            sharex="all",
            sharey="all",
            layout="constrained",
        )
        fig.suptitle(
            f"BLOCK {block_name} | Raw maps | mean freq = {freq_mean:.1f} MHz",
            fontsize=13,
        )
        for row, ant_name in enumerate(batch):
            for col, feed in enumerate(["h", "v"]):
                ax = axes[row, col]
                if ant_name not in ants:
                    add_anchored_text(
                        ax, f"{ant_name} not in observation", loc="center"
                    )
                    ax.set_axis_off()
                else:
                    i_ant = list(ants).index(ant_name)
                    vmin = float(
                        vis_freqmedian.sel(antennas=ant_name).min(
                            dim=("timestamps", "feeds"), skipna=True
                        )
                    )
                    vmax = float(
                        vis_freqmedian.sel(antennas=ant_name).max(
                            dim=("timestamps", "feeds"), skipna=True
                        )
                    )
                    scatter_sky(
                        ax,
                        ds["ra"].isel(antennas=i_ant).values,
                        ds["dec"].isel(antennas=i_ant).values,
                        vis_freqmedian.sel(antennas=ant_name, feeds=feed).values,
                        vmin=vmin,
                        vmax=vmax,
                        cbar=True if feed == "v" else False,
                        point_sources=point_sources,
                    )
                    add_anchored_text(
                        ax, f"{ant_name}{feed}", loc="upper left", alpha=1
                    )
                    if feed == "h":
                        ax.set_ylabel("RA [deg]")
        # Since we are still plotting bad antennas for double checking, there can be a
        # case where the rouge antennas (with outlier pointings) mess up with the auto
        # x/y limits. Thus, we use RA/Dec from good_antennas in this batch to explicitly
        # set the x/y limits for all antennas in this batch.
        good_ants_in_batch = [a for a in batch if a in good_ants][0]
        ra_min = ds["ra"].sel(antennas=good_ants_in_batch).min().values
        ra_max = ds["ra"].sel(antennas=good_ants_in_batch).max().values
        dec_min = ds["dec"].sel(antennas=good_ants_in_batch).min().values
        dec_max = ds["dec"].sel(antennas=good_ants_in_batch).max().values
        axes[0, 0].set_xlim(ra_min, ra_max)
        axes[0, 0].set_ylim(dec_min, dec_max)


plot_raw_maps(ds, flags_before_aoflagger, point_sources)

## 11. Calibrated Scan vs. Synchrotron Model

Compare calibrated data with a smoothed PySM synchrotron model.
Both maps are evaluated on the same antenna track and have their median
removed before comparison. Bright 1 Jy and 5 Jy catalogue sources are
overplotted, and the Spearman correlation between data and model is shown.

In [ ]:
def plot_calibrated_vs_synch(
    ds: xr.Dataset,
    point_sources: dict | None = None,
    freq_plot: float = 730.0,
    freq_half_width: float = 1.5,
    beamsize: float = 57.5,
    beam_ref_freq: float = 1500.0,
    vmin: float = -0.5,
    vmax: float = 1.0,
) -> None:
    ants = ds.coords["antennas"].values
    all_ants = ds.attrs["all_meerkat_antennas"]

    # -- Prepare cal_vis --
    # Load cal_vis, down select to the plot frequency bandwidth and apply combined flags
    bw_cut = xr.ufuncs.logical_and(
        ds.frequencies > (freq_plot - freq_half_width),
        ds.frequencies < (freq_plot + freq_half_width),
    )
    cal_vis_select = (
        ds["cal_vis"].sel(polarisations="I", drop=True).where(bw_cut, drop=True)
    )
    cal_falgs_combined_select = (
        ds["cal_flags_combined"]
        .sel(polarisations="I", drop=True)
        .where(bw_cut, drop=True)
    )
    cal_vis_flagged = cal_vis_select.where(
        xr.ufuncs.logical_not(cal_falgs_combined_select)
    )
    # Calculate median over frequencies and timestamps, skipping NaNs. This will be used for median subtraction.
    cal_vis_flagged_median = cal_vis_flagged.median(
        dim=("timestamps", "frequencies"), skipna=True
    )
    # Down select cal_vis_flagged further for plottiong
    cal_vis_flagged = cal_vis_flagged.sel(frequencies=freq_plot, method="nearest")
    # Median subtraction, broadcast rule should apply
    cal_vis_flagged = cal_vis_flagged - cal_vis_flagged_median

    synch_map = get_synchrotron_map(freq_plot, beamsize, beam_ref_freq)

    # Loop over group of antennas to plot in batches
    for fig_start in range(0, 64, ANTENNAS_PER_FIG):
        batch = all_ants[fig_start : fig_start + ANTENNAS_PER_FIG]

        fig, axes = plt.subplots(
            ANTENNAS_PER_FIG,
            2,
            figsize=(12, 2 * ANTENNAS_PER_FIG),
            sharex="all",
            sharey="all",
            layout="constrained",
        )
        fig.suptitle(
            f"BLOCK {ds.attrs['block_name']} | Cal vs Synch | {freq_plot:.1f} MHz",
            fontsize=13,
        )
        for row, ant_name in enumerate(batch):
            ax_cal, ax_syn = axes[row]
            if ant_name not in ants:
                for ax in (ax_cal, ax_syn):
                    add_anchored_text(
                        ax, f"{ant_name} not in observation", loc="center"
                    )
                    ax.set_axis_off()
                continue

            i_ant = list(ants).index(ant_name)
            ra_ant = ds["ra"].isel(antennas=i_ant).values
            dec_ant = ds["dec"].isel(antennas=i_ant).values
            # if np.all(~np.isfinite(cal_vals)):
            #     for ax in (ax_cal, ax_syn):
            #         add_anchored_text(ax, f"{ant_name} fully masked", loc="center")
            #         ax.set_axis_off()
            #     continue

            synch_vals = get_healpix_interp_vals(synch_map, ra_ant, dec_ant)
            synch_vals = (
                synch_vals - np.median(synch_vals)
            ) / 1e6  # Convert to K_RJ and median subtract
            cal_vis_plot = cal_vis_flagged.sel(antennas=ant_name).values
            r_val = float(ds["r_vis_synch_ant"].sel(antennas=ant_name))

            # Plot maps, shared cbar only on the right column - they should be on the same scale
            scatter_sky(
                ax_cal,
                ra_ant,
                dec_ant,
                cal_vis_plot,
                vmin=vmin,
                vmax=vmax,
                cbar=False,
                point_sources=point_sources,
            )
            scatter_sky(
                ax_syn,
                ra_ant,
                dec_ant,
                synch_vals,
                vmin=vmin,
                vmax=vmax,
                cbar=True,
                point_sources=point_sources,
            )

            add_anchored_text(ax_cal, f"{ant_name}", loc="upper left", alpha=1)
            add_anchored_text(ax_syn, f"r = {r_val:.3f}", loc="upper left", alpha=1)
            if row == 0:
                ax_cal.set_title("Calibrated Vis")
                ax_syn.set_title("Synchrotron model")

        axes[-1, 0].set_xlabel("RA [deg]")
        axes[-1, 1].set_xlabel("RA [deg]")
        for r in range(ANTENNAS_PER_FIG):
            axes[r, 0].set_ylabel("Dec [deg]")
        # Since we are still plotting bad antennas for double checking, there can be a
        # case where the rouge antennas (with outlier pointings) mess up with the auto
        # x/y limits. Thus, we use RA/Dec from good_antennas in this batch to explicitly
        # set the x/y limits for all antennas in this batch.
        good_ants_in_batch = [a for a in batch if a in good_ants][0]
        ra_min = ds["ra"].sel(antennas=good_ants_in_batch).min().values * 1.1
        ra_max = ds["ra"].sel(antennas=good_ants_in_batch).max().values * 1.1
        dec_min = ds["dec"].sel(antennas=good_ants_in_batch).min().values * 1.1
        dec_max = ds["dec"].sel(antennas=good_ants_in_batch).max().values * 1.1
        axes[0, 0].set_xlim(ra_min, ra_max)
        axes[0, 0].set_ylim(dec_min, dec_max)


plot_calibrated_vs_synch(ds, point_sources)